In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")

Groq API Key exists and begins gsk_uqLa


In [3]:
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

GROQ_MODEL = "openai/gpt-oss-120b"

In [4]:
from system_info import retrieve_system_info
system_info = retrieve_system_info()
print(system_info)

{'os': {'system': 'Linux', 'arch': 'x86_64', 'release': '6.6.87.2-microsoft-standard-WSL2', 'version': '#1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025', 'kernel': '6.6.87.2-microsoft-standard-WSL2', 'distro': {'name': 'Ubuntu 24.04.3 LTS', 'version': '24.04'}, 'wsl': True, 'rosetta2_translated': False, 'target_triple': 'x86_64-linux-gnu'}, 'package_managers': ['apt'], 'cpu': {'brand': 'Intel(R) Core(TM) i5-6200U CPU @ 2.30GHz', 'cores_logical': 4, 'cores_physical': 2, 'simd': ['AVX', 'AVX2', 'FMA', 'SSE4_2']}, 'toolchain': {'compilers': {'gcc': 'gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0', 'g++': 'g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0', 'clang': '', 'msvc_cl': ''}, 'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 4.3'}, 'linkers': {'ld_lld': ''}}}


In [5]:
compile_command = ["g++", "-std=c++17", "-O3", "-march=native", "-flto", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [6]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed.
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp.replace('```cpp', '').replace('```', ''))

In [16]:
def port(client, model, python):
    response = client.chat.completions.create(
        model=model, 
        messages=messages_for(python),
        reasoning_effort="medium",
    )
    reply = response.choices[0].message.content
    write_output(reply)
    print("C++ code written to main.cpp")

In [17]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [9]:
def run_python(code):
    exec(code, {"__builtins__": __builtins__})

print("Running Python version:")
run_python(pi)

Running Python version:
Result: 3.141592656089
Execution Time: 52.475609 seconds


In [18]:
port(groq_client, GROQ_MODEL, pi)


C++ code written to main.cpp


In [19]:
def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    result = subprocess.run(run_command, check=True, text=True, capture_output=True)
    print(result.stdout)

print("Compiling and running C++ version:")
compile_and_run()

Compiling and running C++ version:
Result: 3.141592656089
Execution Time: 0.658140 seconds



In [20]:
52.475609/0.658140

79.73320114261404